In [ ]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 3 – Location problems
#  Section: 3.2.1 – The classic p-median problem (PMP-C)
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP            # Modeling language
using HiGHS           # Solver
using CSV             # Data handling
using DataFrames      # Data handling
using Distances       # Distance computations

include("utils/pmp-c_utils.jl")

# Main function to solve p-median problem
function solve_pmp_c(client_file_path, facility_file_path, p)

    # Load latitude and longitude for clients
    client_coordinates = CSV.read(client_file_path, header=true, DataFrame) |> Matrix{Float64}
    n_clients = size(client_coordinates, 1)

    # Load latitude and longitude for facilities
    facility_coordinates = CSV.read(facility_file_path, header=true, DataFrame) |> Matrix{Float64}
    n_facilities = size(facility_coordinates, 1)

    # Use Haversine distance to compute distance matrix
    distance_matrix = Distances.pairwise(Distances.Haversine(), client_coordinates, facility_coordinates, dims=1)

    # Create the model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Decision variables
    @variable(model, y[1:n_facilities], Bin)
    @variable(model, x[1:n_clients, 1:n_facilities], Bin)

    # Objective: minimize total distance
    @objective(model, Min, sum(distance_matrix[i, j] * x[i, j] for i in 1:n_clients, j in 1:n_facilities))

    # Constraint: exactly p facilities selected
    @constraint(model, sum(y) == p)

    # Constraint: each client assigned to one facility
    @constraint(model, [i in 1:n_clients], sum(x[i, j] for j in 1:n_facilities) == 1)

    # Constraint: assignment only to open facilities
    @constraint(model, [i in 1:n_clients, j in 1:n_facilities], x[i, j] <= y[j])

    # Solve the model
    JuMP.optimize!(model)

    # Extract solution
    y_value = JuMP.value.(y)
    x_value = JuMP.value.(x)
    objective_value = JuMP.objective_value(model)

    # Extract / Print selected facility and assignments
    println("Objective value: $(round(objective_value, digits=2)) meters\n")
    facility_ids = findall(y_value .> 0.5)
    println("Selected Facilities (p=$p): $facility_ids\n")
    assignments = Dict(facility_id => findall(x_value[:, facility_id] .> 0.5) for facility_id in facility_ids)
    for (facility_id, clients) in assignments
        println("Facility: $facility_id | Clients: $clients")
    end

    # Plot the solution
    map = plot_solution(client_coordinates, facility_coordinates, assignments, p)
    display(map)
end

# Example usage
solve_pmp_c("data/sjc_pmp_clients.csv", "data/sjc_pmp_facilities.csv", 3)

PyObject <folium.folium.Map object at 0x000001DF5F1F1CD0>

Objective value: 20591.96 meters

Selected Facilities (p=3): [5, 7, 8]

Facility: 5 | Clients: [2, 4, 8, 10, 11, 14, 15, 17, 18, 20, 23, 25, 27, 29, 34, 36, 38, 39, 42, 43, 44, 45, 47, 51, 54, 55, 57, 63, 64, 65, 68, 69, 70, 72, 74, 75, 76, 78, 79, 87, 90, 91, 92]
Facility: 7 | Clients: [3, 13, 19, 21, 31, 33, 41, 50, 52, 56, 59, 62, 67, 77, 83, 89]
Facility: 8 | Clients: [1, 5, 6, 7, 9, 12, 16, 22, 24, 26, 28, 30, 32, 35, 37, 40, 46, 48, 49, 53, 58, 60, 61, 66, 71, 73, 80, 81, 82, 84, 85, 86, 88]
